# 03 — Agente LangGraph Multi-no

Este notebook demonstra o **agente de referencia Geolytics** implementado com [LangGraph](https://github.com/langchain-ai/langgraph). O grafo DAG possui nos especializados para roteamento, decomposicao de perguntas multi-hop, recuperacao RAG, validacao semantica e sintese de respostas.

**Atencao:** para executar com LLM real e necessaria a variavel de ambiente `ANTHROPIC_API_KEY` (ou `OPENAI_API_KEY`). O notebook demonstra a estrutura do grafo mesmo sem chave configurada.

## Pre-requisitos

```bash
pip install langgraph langchain-anthropic langchain-core
# OU para OpenAI:
# pip install langchain-openai
```

Variavel de ambiente (opcional para execucao real):
```bash
export ANTHROPIC_API_KEY='sk-ant-...'
# OU
export OPENAI_API_KEY='sk-...'
```

## Objetivo

1. Importar o modulo `examples/langgraph-agent` via `sys.path`.
2. Construir e inspecionar o grafo compilado.
3. Executar 4 perguntas demo com impressao das transicoes de estado.
4. Renderizar o DAG com `get_graph().draw_mermaid()`.
5. Mostrar como trocar o provedor de LLM via variavel de ambiente.

## 1. Configuracao do ambiente

In [ ]:
import sys
import os

# Adiciona o diretorio do agente ao sys.path
AGENT_DIR = os.path.join(
    os.path.dirname(os.getcwd()), 'examples', 'langgraph-agent'
)
if AGENT_DIR not in sys.path:
    sys.path.insert(0, AGENT_DIR)

# Diretorio raiz do repositorio (para carregar dados)
REPO_ROOT = os.path.dirname(os.getcwd())
os.environ.setdefault('GEOLYTICS_DATA_DIR', os.path.join(REPO_ROOT, 'data'))
os.environ.setdefault('GEOLYTICS_AI_DIR',   os.path.join(REPO_ROOT, 'ai'))

print('AGENT_DIR:', AGENT_DIR)
print('GEOLYTICS_DATA_DIR:', os.environ['GEOLYTICS_DATA_DIR'])
# Saida esperada:
# AGENT_DIR: .../examples/langgraph-agent
# GEOLYTICS_DATA_DIR: .../data

## 2. Importacao dos modulos do agente

In [ ]:
try:
    from state import AgentState, ValidationResult
    print('state.py importado')
except ImportError as exc:
    print(f'Aviso: {exc}')
    # Define stub para demonstracao
    from typing import Any
    from typing_extensions import TypedDict

    class ValidationResult(TypedDict, total=False):
        valid: bool
        violations: list

    class AgentState(TypedDict, total=False):
        question: str
        classification: str
        decomposed: list
        graph_results: list
        rag_results: list
        draft: str
        validation: ValidationResult
        iteration: int
        final_answer: str

    print('Usando stub local do AgentState')
# Saida esperada:
# state.py importado

In [ ]:
# Tenta importar o modulo agent.py
try:
    import agent as geolytics_agent
    print('agent.py importado')
    HAS_AGENT = True
except ImportError as exc:
    print(f'Dependencia ausente: {exc}')
    print('Instale: pip install langgraph langchain-anthropic langchain-core')
    HAS_AGENT = False
# Saida esperada:
# agent.py importado (se langgraph estiver instalado)
# OU: Dependencia ausente: No module named 'langgraph'

## 3. Construcao do grafo compilado

In [ ]:
if HAS_AGENT:
    try:
        compiled = geolytics_agent.build_graph()
        print('Grafo compilado com sucesso')
        print(f'Tipo: {type(compiled)}')
    except Exception as exc:
        print(f'Erro ao compilar grafo: {exc}')
        compiled = None
else:
    compiled = None
    print('Pule esta celula — langgraph nao disponivel')
# Saida esperada (com langgraph instalado):
# Grafo compilado com sucesso
# Tipo: <class 'langgraph.graph.state.CompiledStateGraph'>

## 4. Visualizacao do DAG em Mermaid

In [ ]:
if compiled is not None:
    try:
        mermaid_str = compiled.get_graph().draw_mermaid()
        print('Diagrama Mermaid gerado:')
        print(mermaid_str)
    except Exception as exc:
        print(f'draw_mermaid() nao disponivel: {exc}')
        mermaid_str = None
else:
    mermaid_str = None
    print('Mostrando topologia documentada do agente (ver agent.py):')
# Saida esperada (com langgraph instalado):
# graph TD
#   __start__ --> Router
#   Router --> RAGRetrieve
#   Router --> Decomposer
#   Router --> Validator
#   ...

### Topologia do agente (referencia estatica)

Mesmo sem LangGraph instalado e possivel visualizar a topologia documentada no codigo-fonte:

```mermaid
graph TD
    A([__start__]) --> Router
    Router -->|lookup| RAGRetrieve
    Router -->|multi_hop| Decomposer
    Router -->|validation| Validator
    Decomposer --> GraphQuery
    GraphQuery --> RAGRetrieve
    RAGRetrieve --> Validator
    Validator -->|valid=True ou iter>=2| Synthesizer
    Validator -->|valid=False e iter<2| Synthesizer
    Synthesizer --> B([__end__])
```

## 5. Execucao de 4 perguntas demo

In [ ]:
DEMO_QUESTIONS = [
    {
        'id': 'Q1',
        'descricao': 'Lookup simples',
        'pergunta': 'O que e PAD (Plano de Avaliacao de Descobertas)?',
        'rota_esperada': 'lookup',
    },
    {
        'id': 'Q2',
        'descricao': 'Multi-hop: relacionamento entre entidades',
        'pergunta': 'Quais bacias sedimentares possuem blocos com regime de partilha de producao?',
        'rota_esperada': 'multi_hop',
    },
    {
        'id': 'Q3',
        'descricao': 'Falha de validacao esperada (4P)',
        'pergunta': 'O que e Reserva 4P na classificacao SPE-PRMS?',
        'rota_esperada': 'validation',
    },
    {
        'id': 'Q4',
        'descricao': 'Geomecanica: modulo elastico',
        'pergunta': 'O que e o Modulo de Young e como ele e medido em testemunho?',
        'rota_esperada': 'lookup',
    },
]

for q in DEMO_QUESTIONS:
    print(f"[{q['id']}] {q['descricao']}")
    print(f"    Pergunta: {q['pergunta']}")
    print(f"    Rota esperada: {q['rota_esperada']}")
    print()
# Saida esperada: listagem das 4 perguntas com suas descricoes

In [ ]:
# Executa o agente se disponivel; caso contrario simula as transicoes de estado

def simulate_state_transitions(question: str, route: str) -> dict:
    """Simula as transicoes de estado sem LLM real."""
    state = {'question': question, 'iteration': 0}
    transitions = []

    # Router
    state['classification'] = route
    transitions.append(f'Router -> classificacao: {route}')

    # Nos intermediarios por rota
    if route == 'multi_hop':
        state['decomposed'] = ['subpergunta 1', 'subpergunta 2']
        transitions.append('Decomposer -> ' + str(state['decomposed']))
        state['graph_results'] = [{'id': 'resultado_grafo', 'rel': 'exemplo'}]
        transitions.append('GraphQuery -> graph_results: 1 resultado')
    elif route == 'validation':
        transitions.append('Router -> Validator (sem recuperacao RAG)')

    # RAG (para lookup e multi_hop)
    if route != 'validation':
        state['rag_results'] = [{'id': 'term_pad', 'score': 9.42}]
        transitions.append('RAGRetrieve -> rag_results: 1 chunk')

    # Validator
    state['validation'] = {'valid': route != 'validation', 'violations': []}
    transitions.append(f"Validator -> valid={state['validation']['valid']}")

    # Synthesizer
    state['final_answer'] = f'[Resposta simulada para: {question[:40]}...]'
    transitions.append('Synthesizer -> final_answer')

    state['_transitions'] = transitions
    return state

for q in DEMO_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"[{q['id']}] {q['descricao']}")
    print(f"Pergunta: {q['pergunta']}")
    print()

    if compiled is not None and os.environ.get('ANTHROPIC_API_KEY') or os.environ.get('OPENAI_API_KEY'):
        try:
            result = compiled.invoke({'question': q['pergunta'], 'iteration': 0})
        except Exception as exc:
            print(f'  Erro na execucao real: {exc}')
            result = simulate_state_transitions(q['pergunta'], q['rota_esperada'])
    else:
        result = simulate_state_transitions(q['pergunta'], q['rota_esperada'])

    print('Transicoes de estado:')
    for t in result.get('_transitions', []):
        print(f'  {t}')
    print()
    print(f"Resposta final: {result.get('final_answer', 'N/A')}")
# Saida esperada:
# [Q1] Lookup simples
# Transicoes: Router -> lookup | RAGRetrieve | Validator | Synthesizer
# ...

## 6. Como trocar o provedor de LLM

In [ ]:
# O agente usa a variavel de ambiente LLM_PROVIDER para selecionar o modelo.
# Valores suportados: 'anthropic' (padrao), 'openai'

print('Exemplos de configuracao:')
print()
print('  # Anthropic Claude (padrao):')
print('  export ANTHROPIC_API_KEY="sk-ant-..."')
print('  export LLM_PROVIDER="anthropic"')
print('  export LLM_MODEL="claude-sonnet-4-6"')
print()
print('  # OpenAI:')
print('  export OPENAI_API_KEY="sk-..."')
print('  export LLM_PROVIDER="openai"')
print('  export LLM_MODEL="gpt-4o"')
print()
print('  # Dentro do notebook:')
print('  os.environ["LLM_PROVIDER"] = "openai"')
print('  os.environ["OPENAI_API_KEY"] = "sk-..."')
print('  compiled = geolytics_agent.build_graph()  # reconstroi com novo provider')
# Esta celula e apenas informativa e nao executa nenhuma chamada de API

## Proximo passo

Veja o notebook **04_geomec_qa.ipynb** para um estudo de caso aprofundado em geomecanica: circulo de Mohr, validacao SHACL e mapeamento de fraturas para o GSO.